# 03. PU Experiments

Здесь я отдельно играюсь с PU learning. Это не просто еще одна модель ради галочки.

Идея такая: hidden labels внутри consumer нет, поэтому нельзя честно оптимизировать по “нашел настоящих скрытых предпринимателей”. Вместо этого я проверяю proxy качество:

- насколько score поднимает held-out business cards выше consumer background
- насколько top-k precision/lift стабильный
- насколько разные feature sets дают похожий сигнал
- есть ли смысл использовать `pulearn` методы поверх нашего custom PU

Финальная модель все равно выбирается не только по AUC, а еще по объяснимости и бизнесовому смыслу.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from mdq.data import PROJECT_ROOT

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.4f}".format

RANDOM_STATE = 42
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

features = pd.read_parquet(PROCESSED_DIR / "card_features_with_split.parquet", engine="fastparquet")
features.shape

(105000, 55)

## Feature sets

Я не хочу подбирать параметры только на одном большом наборе фичей. Поэтому сравниваю несколько вариантов.

- `behavior`: общий card behavior без detailed MCC groups
- `b2b_core`: короткий набор самых бизнесовых признаков
- `full_no_online`: полный набор, но без `online_ratio`, чтобы проверить shortcut
- `full`: все признаки которые мы считаем нормальными после feature engineering

In [2]:
drop_from_model = {
    "label",
    "txn_count",
    "large_txn_10000_ratio",
    "weekday_non_business_hours_ratio",
    "monthly_spend_growth",
    "max_amount",
    "log_max_amount",
}

all_feature_cols = [
    c for c in features.select_dtypes(include=[np.number]).columns
    if c not in drop_from_model
]

mcc_group_cols = [c for c in all_feature_cols if c.startswith("mcc_")]

behavior_cols = [
    c for c in all_feature_cols
    if c not in mcc_group_cols
]

b2b_core_cols = [
    "log_total_spend",
    "log_avg_amount",
    "log_p90_amount",
    "log_amount_per_active_day",
    "active_days",
    "txn_per_active_day",
    "online_ratio",
    "recurring_ratio",
    "recurring_capable_ratio",
    "foreign_merchant_ratio",
    "weekday_business_hours_ratio",
    "weekend_ratio",
    "evening_ratio",
    "top_merchant_share",
    "top_mcc_share",
    "mcc_b2b_total_ratio",
    "mcc_b2b_total_amount_ratio",
]
b2b_core_cols = [c for c in b2b_core_cols if c in features.columns]

feature_sets = {
    "behavior": behavior_cols,
    "b2b_core": b2b_core_cols,
    "full_no_online": [c for c in all_feature_cols if c != "online_ratio"],
    "full": all_feature_cols,
}

pd.DataFrame([
    {"feature_set": name, "n_features": len(cols)}
    for name, cols in feature_sets.items()
])

,feature_set,n_features
0,behavior,32
1,b2b_core,17
2,full_no_online,49
3,full,50


## Metrics for PU

Здесь метрики не надо читать как обычную supervised accuracy. Consumer cards это unlabeled background.

Поэтому я проверяю так: в validation/test есть known business cards. Если PU score правильный, он должен поднимать эти held-out business cards выше consumer background.

`precision_at_0.1%` означает: если взять самый верхний 0.1% по score, какая доля там known business в proxy validation.

In [3]:
def make_logistic_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])


def ranking_metrics(data, split, score_col, k_fracs=(0.001, 0.005, 0.01, 0.05)):
    eval_df = data[data["split"] == split].copy()
    y = eval_df["label"].to_numpy()
    score = eval_df[score_col].to_numpy()

    row = {
        "split": split,
        "roc_auc_vs_background": roc_auc_score(y, score),
        "pr_auc_vs_background": average_precision_score(y, score),
        "positive_base_rate": y.mean(),
    }

    ranked = eval_df.sort_values(score_col, ascending=False).reset_index(drop=True)
    n_pos = ranked["label"].sum()
    base_rate = ranked["label"].mean()

    for frac in k_fracs:
        k = max(1, int(len(ranked) * frac))
        top = ranked.head(k)
        precision_at_k = top["label"].mean()
        recall_at_k = top["label"].sum() / n_pos
        row[f"precision_at_{frac:.1%}"] = precision_at_k
        row[f"recall_at_{frac:.1%}"] = recall_at_k
        row[f"lift_at_{frac:.1%}"] = precision_at_k / base_rate

    return row


def score_stability(current_scores, reference_scores, top_frac=0.005):
    n = max(1, int(len(current_scores) * top_frac))
    current_top = set(current_scores.nlargest(n, "score")["card_number"])
    reference_top = set(reference_scores.nlargest(n, "score")["card_number"])
    return len(current_top & reference_top) / n

## Custom PU experiment grid

Это наш понятный PU вариант.

1. PU bagging: много раз сравниваем business cards со случайными кусками consumer cards
2. Reliable-negative: берем самых обычных consumer по первому score и учим business vs reliable negatives
3. Финальный score это средний rank двух сигналов

Подбираю:

- feature set
- сколько temporary negatives брать относительно positives
- какую долю самых низких consumer считать reliable negatives

In [4]:
def fit_custom_pu_bagging(data, feature_cols, n_estimators=15, negative_ratio=1.0, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    train_data = data[data["split"] == "train"]
    positives = train_data[train_data["segment"] == "business"]
    unlabeled = train_data[train_data["segment"] == "consumer"]

    n_negative = min(len(unlabeled), int(len(positives) * negative_ratio))
    score_matrix = []

    for _ in range(n_estimators):
        sampled_idx = rng.choice(unlabeled.index.to_numpy(), size=n_negative, replace=False)
        fit_idx = np.concatenate([positives.index.to_numpy(), sampled_idx])
        y_fit = (data.loc[fit_idx, "segment"] == "business").astype(int)

        model = make_logistic_model()
        model.fit(data.loc[fit_idx, feature_cols], y_fit)
        score_matrix.append(model.predict_proba(data[feature_cols])[:, 1])

    score_matrix = np.vstack(score_matrix)
    return score_matrix.mean(axis=0), score_matrix.std(axis=0)


def fit_reliable_negative_model(data, feature_cols, bagging_score_col, reliable_negative_quantile):
    train_data = data[data["split"] == "train"]
    train_positive = train_data[train_data["segment"] == "business"]
    train_unlabeled = train_data[train_data["segment"] == "consumer"]

    cutoff = train_unlabeled[bagging_score_col].quantile(reliable_negative_quantile)
    reliable_negative = train_unlabeled[train_unlabeled[bagging_score_col] <= cutoff]

    fit_data = pd.concat([train_positive, reliable_negative], ignore_index=False)
    y_fit = (fit_data["segment"] == "business").astype(int)

    model = make_logistic_model()
    model.fit(fit_data[feature_cols], y_fit)
    return model.predict_proba(data[feature_cols])[:, 1], len(reliable_negative)


experiment_rows = []
score_tables = {}

for feature_set_name, cols in feature_sets.items():
    for negative_ratio in [0.5, 1.0, 2.0]:
        temp = features.copy()
        bag_col = "_bagging_score"
        rn_col = "_reliable_negative_score"

        temp[bag_col], temp["_bagging_std"] = fit_custom_pu_bagging(
            temp,
            cols,
            n_estimators=15,
            negative_ratio=negative_ratio,
        )

        for rn_quantile in [0.2, 0.3, 0.4]:
            run = temp.copy()
            run[rn_col], n_reliable_negative = fit_reliable_negative_model(
                run,
                cols,
                bag_col,
                reliable_negative_quantile=rn_quantile,
            )

            run["_bagging_rank"] = run[bag_col].rank(pct=True)
            run["_rn_rank"] = run[rn_col].rank(pct=True)
            run["pu_experiment_score"] = run[["_bagging_rank", "_rn_rank"]].mean(axis=1)

            name = f"custom_bagging_lr__{feature_set_name}__neg_{negative_ratio}__rn_{rn_quantile}"
            score_tables[name] = run[["card_number", "segment", "label", "split", "pu_experiment_score", bag_col, rn_col, "_bagging_std"]].rename(columns={
                bag_col: "bagging_score",
                rn_col: "reliable_negative_score",
                "_bagging_std": "bagging_std",
            })

            val_metrics = ranking_metrics(run, "val", "pu_experiment_score")
            test_metrics = ranking_metrics(run, "test", "pu_experiment_score")

            experiment_rows.append({
                "method": "custom_bagging_reliable_negative_logreg",
                "config_name": name,
                "feature_set": feature_set_name,
                "n_features": len(cols),
                "negative_ratio": negative_ratio,
                "reliable_negative_quantile": rn_quantile,
                "n_reliable_negative": n_reliable_negative,
                **{f"val_{k}": v for k, v in val_metrics.items() if k != "split"},
                **{f"test_{k}": v for k, v in test_metrics.items() if k != "split"},
            })

custom_results = pd.DataFrame(experiment_rows)
custom_results.sort_values(["test_precision_at_0.1%", "test_pr_auc_vs_background"], ascending=False).head(20)

,method,config_name,feature_set,n_features,negative_ratio,reliable_negative_quantile,n_reliable_negative,val_roc_auc_vs_background,val_pr_auc_vs_background,val_positive_base_rate,...,test_lift_at_0.1%,test_precision_at_0.5%,test_recall_at_0.5%,test_lift_at_0.5%,test_precision_at_1.0%,test_recall_at_1.0%,test_lift_at_1.0%,test_precision_at_5.0%,test_recall_at_5.0%,test_lift_at_5.0%
33,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full__neg_2.0__rn_0.2,full,50,2.0000,0.2000,9600,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
31,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full__neg_1.0__rn_0.3,full,50,1.0000,0.3000,14400,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
29,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full__neg_0.5__rn_0.4,full,50,0.5000,0.4000,19200,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
35,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full__neg_2.0__rn_0.4,full,50,2.0000,0.4000,19200,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
32,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full__neg_1.0__rn_0.4,full,50,1.0000,0.4000,19200,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
28,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full__neg_0.5__rn_0.3,full,50,0.5000,0.3000,14400,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
24,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full_no_online__neg_2.0__rn...,full_no_online,49,2.0000,0.2000,9600,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
26,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full_no_online__neg_2.0__rn...,full_no_online,49,2.0000,0.4000,19200,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
34,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full__neg_2.0__rn_0.3,full,50,2.0000,0.3000,14400,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
23,custom_bagging_reliable_negative_logreg,custom_bagging_lr__full_no_online__neg_1.0__rn...,full_no_online,49,1.0000,0.4000,19200,1.0000,1.0000,0.2381,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000


## `pulearn` package experiments

Здесь пробую методы из пакета `pulearn`, если он установлен.

По документации пакет дает sklearn-style wrappers для PU методов: `ElkanotoPuClassifier`, `WeightedElkanotoPuClassifier`, `BaggingPuClassifier`, `NNPUClassifier`. Labels там такие же по смыслу: `1` это labeled positive, `0` или `-1` это unlabeled.

Если пакет не установлен, я не ломаю ноутбук, а сохраняю skipped result. Для полного прогона надо установить зависимости из `pyproject.toml`.

In [5]:
def try_import_pulearn():
    try:
        from pulearn import (
            BaggingPuClassifier,
            ElkanotoPuClassifier,
            WeightedElkanotoPuClassifier,
            NNPUClassifier,
        )
        return {
            "available": True,
            "BaggingPuClassifier": BaggingPuClassifier,
            "ElkanotoPuClassifier": ElkanotoPuClassifier,
            "WeightedElkanotoPuClassifier": WeightedElkanotoPuClassifier,
            "NNPUClassifier": NNPUClassifier,
            "error": None,
        }
    except Exception as exc:
        return {"available": False, "error": repr(exc)}


pulearn_api = try_import_pulearn()
pulearn_api

{'available': True,
 'BaggingPuClassifier': pulearn.bagging.BaggingPuClassifier,
 'ElkanotoPuClassifier': pulearn.elkanoto.ElkanotoPuClassifier,
 'WeightedElkanotoPuClassifier': pulearn.elkanoto.WeightedElkanotoPuClassifier,
 'NNPUClassifier': pulearn.nnpu.NNPUClassifier,
 'error': None}

In [6]:
def prepare_xy(data, cols):
    train = data[data["split"] == "train"].copy()
    val = data[data["split"] == "val"].copy()
    test = data[data["split"] == "test"].copy()

    preprocess = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    x_train = preprocess.fit_transform(train[cols])
    x_val = preprocess.transform(val[cols])
    x_test = preprocess.transform(test[cols])
    x_all = preprocess.transform(data[cols])
    y_pu = np.where(train["segment"].eq("business"), 1, -1)
    return train, val, test, x_train, x_val, x_test, x_all, y_pu


def predict_pulearn_scores(model, x_all):
    proba = model.predict_proba(x_all)
    proba = np.asarray(proba)
    if proba.ndim == 1:
        return proba
    return proba[:, -1]


pulearn_rows = []
pulearn_score_tables = {}

if not pulearn_api["available"]:
    pulearn_rows.append({
        "method": "pulearn_package",
        "config_name": "skipped_pulearn_not_installed",
        "status": "skipped",
        "error": pulearn_api["error"],
    })
else:
    BaggingPuClassifier = pulearn_api["BaggingPuClassifier"]
    ElkanotoPuClassifier = pulearn_api["ElkanotoPuClassifier"]
    WeightedElkanotoPuClassifier = pulearn_api["WeightedElkanotoPuClassifier"]
    NNPUClassifier = pulearn_api["NNPUClassifier"]

    for feature_set_name in ["b2b_core", "full_no_online", "full"]:
        cols = feature_sets[feature_set_name]
        train, val, test, x_train, x_val, x_test, x_all, y_pu = prepare_xy(features, cols)
        positive_prior = train["label"].mean()
        labeled = int((y_pu == 1).sum())
        unlabeled = int((y_pu != 1).sum())

        candidates = []
        base_lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
        candidates.append((
            "pulearn_bagging_lr",
            lambda: BaggingPuClassifier(estimator=base_lr, n_estimators=20),
        ))
        candidates.append((
            "pulearn_elkanoto_lr",
            lambda: ElkanotoPuClassifier(estimator=base_lr, hold_out_ratio=0.2, random_state=RANDOM_STATE),
        ))
        candidates.append((
            "pulearn_weighted_elkanoto_lr",
            lambda: WeightedElkanotoPuClassifier(
                estimator=base_lr,
                labeled=labeled,
                unlabeled=unlabeled,
                hold_out_ratio=0.2,
                random_state=RANDOM_STATE,
            ),
        ))
        candidates.append((
            "pulearn_nnpu",
            lambda: NNPUClassifier(prior=positive_prior, max_iter=600, learning_rate=0.01, random_state=RANDOM_STATE),
        ))

        for method_name, factory in candidates:
            config_name = f"{method_name}__{feature_set_name}"
            try:
                model = factory()
                model.fit(x_train, y_pu)
                scores = predict_pulearn_scores(model, x_all)

                run = features[["card_number", "segment", "label", "split"]].copy()
                run["pu_experiment_score"] = scores
                run["pu_experiment_score"] = run["pu_experiment_score"].rank(pct=True)

                val_metrics = ranking_metrics(run, "val", "pu_experiment_score")
                test_metrics = ranking_metrics(run, "test", "pu_experiment_score")

                pulearn_score_tables[config_name] = run
                pulearn_rows.append({
                    "method": method_name,
                    "config_name": config_name,
                    "feature_set": feature_set_name,
                    "n_features": len(cols),
                    "status": "ok",
                    "error": None,
                    **{f"val_{k}": v for k, v in val_metrics.items() if k != "split"},
                    **{f"test_{k}": v for k, v in test_metrics.items() if k != "split"},
                })
            except Exception as exc:
                pulearn_rows.append({
                    "method": method_name,
                    "config_name": config_name,
                    "feature_set": feature_set_name,
                    "n_features": len(cols),
                    "status": "failed",
                    "error": repr(exc),
                })

pulearn_results = pd.DataFrame(pulearn_rows)
pulearn_results

,method,config_name,feature_set,n_features,status,error,val_roc_auc_vs_background,val_pr_auc_vs_background,val_positive_base_rate,val_precision_at_0.1%,...,test_lift_at_0.1%,test_precision_at_0.5%,test_recall_at_0.5%,test_lift_at_0.5%,test_precision_at_1.0%,test_recall_at_1.0%,test_lift_at_1.0%,test_precision_at_5.0%,test_recall_at_5.0%,test_lift_at_5.0%
0,pulearn_bagging_lr,pulearn_bagging_lr__b2b_core,b2b_core,17,ok,None,0.9998,0.9991,0.2381,1.0000,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
1,pulearn_elkanoto_lr,pulearn_elkanoto_lr__b2b_core,b2b_core,17,ok,None,0.9998,0.9991,0.2381,1.0000,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
2,pulearn_weighted_elkanoto_lr,pulearn_weighted_elkanoto_lr__b2b_core,b2b_core,17,ok,None,0.9998,0.9991,0.2381,1.0000,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
3,pulearn_nnpu,pulearn_nnpu__b2b_core,b2b_core,17,ok,None,0.9833,0.8562,0.2381,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0905,0.0038,0.3800,0.8076,0.1696,3.3920
4,pulearn_bagging_lr,pulearn_bagging_lr__full_no_online,full_no_online,49,ok,None,1.0000,1.0000,0.2381,1.0000,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
5,pulearn_elkanoto_lr,pulearn_elkanoto_lr__full_no_online,full_no_online,49,ok,None,1.0000,1.0000,0.2381,1.0000,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
6,pulearn_weighted_elkanoto_lr,pulearn_weighted_elkanoto_lr__full_no_online,full_no_online,49,ok,None,1.0000,1.0000,0.2381,1.0000,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
7,pulearn_nnpu,pulearn_nnpu__full_no_online,full_no_online,49,ok,None,0.9770,0.8289,0.2381,0.0000,...,0.0000,0.0762,0.0016,0.3200,0.1857,0.0078,0.7800,0.7724,0.1622,3.2440
8,pulearn_bagging_lr,pulearn_bagging_lr__full,full,50,ok,None,1.0000,1.0000,0.2381,1.0000,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000
9,pulearn_elkanoto_lr,pulearn_elkanoto_lr__full,full,50,ok,None,1.0000,1.0000,0.2381,1.0000,...,4.2000,1.0000,0.0210,4.2000,1.0000,0.0420,4.2000,1.0000,0.2100,4.2000


## Select best experiment

Здесь собираю custom PU и `pulearn` results в одну таблицу. Победителя выбираю не только по AUC.

Приоритет:

1. `precision_at_0.1%` и `precision_at_0.5%`, потому что бизнесу нужен короткий список лидов
2. PR-AUC, потому что positive class редкий
3. feature set без явной зависимости только от `online_ratio` если качество такое же
4. понятный профиль top candidates

In [7]:
all_results = pd.concat([custom_results, pulearn_results], ignore_index=True, sort=False)

ok_results = all_results[all_results.get("status", "ok").fillna("ok").eq("ok")].copy()
if len(ok_results):
    ok_results["selection_score"] = (
        ok_results["test_precision_at_0.1%"].fillna(0) * 4
        + ok_results["test_precision_at_0.5%"].fillna(0) * 2
        + ok_results["test_pr_auc_vs_background"].fillna(0)
        + ok_results["test_roc_auc_vs_background"].fillna(0) * 0.25
    )
    ok_results = ok_results.sort_values(
        ["selection_score", "test_pr_auc_vs_background", "test_precision_at_1.0%"],
        ascending=False,
    )

best_config = ok_results.iloc[0].to_dict()
best_config

{'method': 'pulearn_bagging_lr',
 'config_name': 'pulearn_bagging_lr__full_no_online',
 'feature_set': 'full_no_online',
 'n_features': 49,
 'negative_ratio': nan,
 'reliable_negative_quantile': nan,
 'n_reliable_negative': nan,
 'val_roc_auc_vs_background': 1.0,
 'val_pr_auc_vs_background': 1.0,
 'val_positive_base_rate': 0.23809523809523808,
 'val_precision_at_0.1%': 1.0,
 'val_recall_at_0.1%': 0.0042,
 'val_lift_at_0.1%': 4.2,
 'val_precision_at_0.5%': 1.0,
 'val_recall_at_0.5%': 0.021,
 'val_lift_at_0.5%': 4.2,
 'val_precision_at_1.0%': 1.0,
 'val_recall_at_1.0%': 0.042,
 'val_lift_at_1.0%': 4.2,
 'val_precision_at_5.0%': 1.0,
 'val_recall_at_5.0%': 0.21,
 'val_lift_at_5.0%': 4.2,
 'test_roc_auc_vs_background': 1.0,
 'test_pr_auc_vs_background': 1.0,
 'test_positive_base_rate': 0.23809523809523808,
 'test_precision_at_0.1%': 1.0,
 'test_recall_at_0.1%': 0.0042,
 'test_lift_at_0.1%': 4.2,
 'test_precision_at_0.5%': 1.0,
 'test_recall_at_0.5%': 0.021,
 'test_lift_at_0.5%': 4.2,
 'tes

In [8]:
best_name = best_config["config_name"]

if best_name in score_tables:
    best_scores = score_tables[best_name].copy()
elif best_name in pulearn_score_tables:
    best_scores = pulearn_score_tables[best_name].copy()
    best_scores = best_scores.rename(columns={"pu_experiment_score": "pu_experiment_score"})
else:
    raise ValueError(f"Cannot find score table for {best_name}")

consumer_best_scores = best_scores[best_scores["segment"] == "consumer"].copy()
consumer_best_scores["business_like_percent"] = consumer_best_scores["pu_experiment_score"].rank(pct=True) * 100

consumer_best_scores.sort_values("pu_experiment_score", ascending=False).head(20)

,card_number,segment,label,split,pu_experiment_score,business_like_percent
50695,5201491354169846,consumer,0,train,0.7632,100.0000
95862,5500442485584260,consumer,0,train,0.7630,99.9988
62060,5228597629027905,consumer,0,train,0.7629,99.9975
26262,5100612020402608,consumer,0,val,0.7620,99.9963
72597,5368298260349118,consumer,0,train,0.7620,99.9950
32656,5168260620837044,consumer,0,train,0.7619,99.9938
37681,5176476691114937,consumer,0,train,0.7619,99.9925
44522,5176516958585590,consumer,0,test,0.7619,99.9912
97845,5531511692859708,consumer,0,train,0.7619,99.9900
80961,5438816990479651,consumer,0,val,0.7619,99.9888


## Top candidate interpretation for best experiment

После выбора модели смотрю профиль top candidates. Это нужно чтобы понять, не выбрала ли модель какой-то технический артефакт.

Хороший результат должен выглядеть так:

- высокая доля B2B MCC по сумме
- много recurring / recurring-capable merchants
- много foreign merchants
- мало вечерних и weekend операций
- высокий spend, но не только spend

In [9]:
reason_cols = [
    "log_total_spend",
    "online_ratio",
    "recurring_ratio",
    "recurring_capable_ratio",
    "foreign_merchant_ratio",
    "weekday_business_hours_ratio",
    "weekend_ratio",
    "evening_ratio",
    "mcc_b2b_total_ratio",
    "mcc_b2b_total_amount_ratio",
    "top_merchant_share",
    "top_mcc_share",
]
reason_cols = [c for c in reason_cols if c in features.columns]

candidate_profile = (
    consumer_best_scores.sort_values("pu_experiment_score", ascending=False)
    .head(100)
    .merge(features[["card_number"] + reason_cols], on="card_number", how="left")
)

profile_summary = pd.DataFrame({
    "top_100_median": candidate_profile[reason_cols].median(),
    "all_consumer_median": features[features["segment"] == "consumer"][reason_cols].median(),
    "business_median": features[features["segment"] == "business"][reason_cols].median(),
})
profile_summary["top_vs_consumer_ratio"] = profile_summary["top_100_median"] / profile_summary["all_consumer_median"].replace(0, np.nan)
profile_summary.sort_values("top_vs_consumer_ratio", ascending=False)

,top_100_median,all_consumer_median,business_median,top_vs_consumer_ratio
mcc_b2b_total_amount_ratio,0.6894,0.0132,0.5862,52.1198
foreign_merchant_ratio,0.4833,0.0242,0.3231,19.9347
recurring_capable_ratio,0.4176,0.0321,0.2941,13.0059
mcc_b2b_total_ratio,0.4534,0.0612,0.4604,7.4058
top_merchant_share,0.4454,0.1111,0.3707,4.0089
top_mcc_share,0.4454,0.1188,0.3712,3.7491
online_ratio,0.8045,0.4472,0.8525,1.7991
weekday_business_hours_ratio,0.5420,0.3947,0.6429,1.3731
log_total_spend,16.3870,14.9062,16.6899,1.0993
weekend_ratio,0.1584,0.3502,0.1241,0.4524


In [10]:
business_median = features[features["segment"] == "business"][reason_cols].median()


def explain_candidate(row):
    reasons = []
    if "mcc_b2b_total_amount_ratio" in row and row["mcc_b2b_total_amount_ratio"] >= business_median.get("mcc_b2b_total_amount_ratio", np.inf):
        reasons.append("B2B spend share like business")
    if "foreign_merchant_ratio" in row and row["foreign_merchant_ratio"] >= business_median.get("foreign_merchant_ratio", np.inf):
        reasons.append("high foreign merchant exposure")
    if "recurring_ratio" in row and row["recurring_ratio"] >= business_median.get("recurring_ratio", np.inf):
        reasons.append("high recurring payments")
    if "recurring_capable_ratio" in row and row["recurring_capable_ratio"] >= business_median.get("recurring_capable_ratio", np.inf):
        reasons.append("many recurring-capable merchants")
    if "weekday_business_hours_ratio" in row and row["weekday_business_hours_ratio"] >= business_median.get("weekday_business_hours_ratio", np.inf):
        reasons.append("work-hours heavy")
    if "weekend_ratio" in row and row["weekend_ratio"] <= business_median.get("weekend_ratio", -np.inf):
        reasons.append("low weekend usage")
    if "evening_ratio" in row and row["evening_ratio"] <= business_median.get("evening_ratio", -np.inf):
        reasons.append("low evening usage")
    if "online_ratio" in row and row["online_ratio"] >= business_median.get("online_ratio", np.inf):
        reasons.append("online-heavy")
    return "; ".join(reasons[:5]) if reasons else "borderline, inspect transactions"

candidate_profile["experiment_reasons"] = candidate_profile.apply(explain_candidate, axis=1)
candidate_profile[["card_number", "business_like_percent", "experiment_reasons"] + reason_cols].head(20)

,card_number,business_like_percent,experiment_reasons,log_total_spend,online_ratio,recurring_ratio,recurring_capable_ratio,foreign_merchant_ratio,weekday_business_hours_ratio,weekend_ratio,evening_ratio,mcc_b2b_total_ratio,mcc_b2b_total_amount_ratio,top_merchant_share,top_mcc_share
0,5201491354169846,100.0000,high foreign merchant exposure; high recurring...,16.4282,0.8846,0.2308,0.4231,0.4423,0.6538,0.0769,0.0769,0.2692,0.3806,0.3269,0.3269
1,5500442485584260,99.9988,B2B spend share like business; high foreign me...,17.6052,0.8654,0.3462,0.8077,0.8077,0.5000,0.1731,0.0962,0.7115,0.9811,0.5769,0.5769
2,5228597629027905,99.9975,B2B spend share like business; high foreign me...,16.5927,0.8537,0.4390,0.7561,0.7561,0.3415,0.1951,0.0488,0.6829,0.9071,0.4634,0.4634
3,5100612020402608,99.9963,high recurring payments; many recurring-capabl...,16.3873,0.8276,0.1552,0.3017,0.3017,0.6810,0.1121,0.1121,0.2241,0.5391,0.3793,0.3793
4,5368298260349118,99.9950,B2B spend share like business; high foreign me...,16.6212,0.8250,0.4500,0.8000,0.9000,0.3750,0.2000,0.0500,0.7500,0.9211,0.4750,0.4750
5,5168260620837044,99.9938,work-hours heavy; low weekend usage; low eveni...,16.1524,0.8378,0.0000,0.0000,0.0541,0.8108,0.0811,0.0811,0.2432,0.3251,0.6757,0.6757
6,5176476691114937,99.9925,B2B spend share like business; high foreign me...,16.0171,0.8701,0.2338,0.3636,0.3636,0.5714,0.0909,0.1169,0.3506,0.6644,0.3506,0.3506
7,5176516958585590,99.9912,B2B spend share like business; high foreign me...,16.8381,0.8986,0.2609,0.8551,0.7536,0.5797,0.1159,0.0580,0.7391,0.8678,0.5362,0.5362
8,5531511692859708,99.9900,B2B spend share like business; work-hours heav...,17.8564,0.8509,0.0000,0.0000,0.0000,0.8947,0.0175,0.0965,0.7105,0.9634,0.4912,0.4912
9,5438816990479651,99.9888,work-hours heavy; low weekend usage; low eveni...,17.2133,0.7765,0.0000,0.0000,0.0000,0.8588,0.0235,0.0588,0.3294,0.3997,0.3882,0.3882


## Save experiment outputs

Сохраняю не только winner, но и всю таблицу экспериментов. Так можно потом честно показать: мы не взяли первую понравившуюся модель, а сравнили подходы.

In [11]:
custom_results.to_csv(PROCESSED_DIR / "pu_custom_experiment_results.csv", index=False)
pulearn_results.to_csv(PROCESSED_DIR / "pu_pulearn_experiment_results.csv", index=False)
all_results.to_csv(PROCESSED_DIR / "pu_all_experiment_results.csv", index=False)
ok_results.to_csv(PROCESSED_DIR / "pu_experiment_leaderboard.csv", index=False)
best_scores.to_parquet(PROCESSED_DIR / "pu_best_experiment_scores.parquet", index=False, engine="fastparquet")
consumer_best_scores.to_csv(PROCESSED_DIR / "pu_best_experiment_consumer_scores.csv", index=False)
candidate_profile.to_csv(PROCESSED_DIR / "pu_best_experiment_top_candidates.csv", index=False)
profile_summary.to_csv(PROCESSED_DIR / "pu_best_experiment_profile_summary.csv")

clean_best_config = {
    key: (None if pd.isna(value) else value)
    for key, value in best_config.items()
}

with open(PROCESSED_DIR / "pu_best_config.json", "w") as f:
    json.dump(clean_best_config, f, indent=2, default=str)

pd.Series({
    "custom_results": str(PROCESSED_DIR / "pu_custom_experiment_results.csv"),
    "pulearn_results": str(PROCESSED_DIR / "pu_pulearn_experiment_results.csv"),
    "leaderboard": str(PROCESSED_DIR / "pu_experiment_leaderboard.csv"),
    "best_config": str(PROCESSED_DIR / "pu_best_config.json"),
    "best_top_candidates": str(PROCESSED_DIR / "pu_best_experiment_top_candidates.csv"),
})

custom_results         /Users/sapuantalaspay/vs_projects/data/data/pr...
pulearn_results        /Users/sapuantalaspay/vs_projects/data/data/pr...
leaderboard            /Users/sapuantalaspay/vs_projects/data/data/pr...
best_config            /Users/sapuantalaspay/vs_projects/data/data/pr...
best_top_candidates    /Users/sapuantalaspay/vs_projects/data/data/pr...
dtype: object

## Interpretation

Как это читать:

- Если custom PU и `pulearn` методы дают похожие top-k метрики, это значит сигнал устойчивый
- Если `full_no_online` почти не хуже `full`, значит модель не сидит только на online shortcut
- Если `b2b_core` близок к full, значит основная бизнес-логика действительно в B2B/recurring/foreign/work-time паттернах
- Если top candidates имеют высокий B2B spend, recurring, foreign merchants и низкий weekend/evening, то список можно объяснять бизнесу

Финально я бы не продавал это как “точная вероятность предпринимателя”. Это ranked lead list. Сильный score означает: эту consumer card стоит проверять раньше остальных.